In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
import warnings
warnings.filterwarnings('ignore')

spark = (SparkSession.builder
    .appName("FeatureEngineering")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.driver.memory", "2g")
    .getOrCreate())

spark.sparkContext.setLogLevel("WARN")

# Silver tablosunu oku
SILVER_PATH = "/home/jovyan/work/delta-lake/silver/climate_events"
df = spark.read.format("delta").load(SILVER_PATH)

print(f"Silver tablosu yüklendi: {df.count():,} satır, {len(df.columns)} sütun")
print("Spark Session hazır ✅")

Silver tablosu yüklendi: 5,000 satır, 22 sütun
Spark Session hazır ✅


In [2]:
# ============================================================
# FEATURE GROUP 1: SICAKLIK TÜREVLERI
# ============================================================
# Bu grup, ham sıcaklık değerinden ML için anlamlı sinyaller üretir.
# ML modelleri ham değerden çok, kategorize edilmiş bilgiyi daha iyi öğrenir.

df_features = df

# 1. temp_range_c: Günlük sıcaklık farkı (max - min)
# İŞ MANTIĞI: Geniş sıcaklık farkı kıtasal iklimi, dar fark deniz iklimini
# işaret eder. Hava durumu tahmininde kritik bir sinyal.
df_features = df_features.withColumn(
    "temp_range_c",
    col("max_temp_c") - col("min_temp_c")
)

# 2. is_freezing: Donma noktası altı mı?
# İŞ MANTIĞI: Sıcaklığın ham değeri yerine "kritik eşik" bilgisi
# don olayları, kar yağışı tahmini gibi sınıflandırmalarda daha güçlü.
df_features = df_features.withColumn(
    "is_freezing",
    when(col("avg_temp_c") < 0, 1).otherwise(0)
)

# 3. is_hot_day: Sıcak gün mü? (>30°C)
# İŞ MANTIĞI: Aşırı sıcak günler ayrı bir hava durumu kategorisidir,
# enerji tüketimi tahmini, sağlık uyarıları için önemli.
df_features = df_features.withColumn(
    "is_hot_day",
    when(col("avg_temp_c") > 30, 1).otherwise(0)
)

print("✅ Sıcaklık feature'ları üretildi: temp_range_c, is_freezing, is_hot_day")
df_features.select("avg_temp_c", "min_temp_c", "max_temp_c", 
                   "temp_range_c", "is_freezing", "is_hot_day").show(5)

✅ Sıcaklık feature'ları üretildi: temp_range_c, is_freezing, is_hot_day
+----------+----------+----------+------------+-----------+----------+
|avg_temp_c|min_temp_c|max_temp_c|temp_range_c|is_freezing|is_hot_day|
+----------+----------+----------+------------+-----------+----------+
|      27.3|      NULL|      NULL|        NULL|          0|         0|
|      26.2|      22.0|      34.0|        12.0|          0|         0|
|      28.8|      23.0|      36.0|        13.0|          0|         0|
|      27.8|      20.0|      NULL|        NULL|          0|         0|
|      27.3|      19.0|      NULL|        NULL|          0|         0|
+----------+----------+----------+------------+-----------+----------+
only showing top 5 rows



In [3]:
# ============================================================
# FEATURE GROUP 2: YAĞIŞ KATEGORİZASYONU
# ============================================================

# 4. precipitation_category: Yağış miktarına göre kategori
# İŞ MANTIĞI: Sürekli yağış değeri yerine kategorik gösterim,
# ML modelleri için daha güçlü sinyal verir. Meteorolojide standart eşikler:
#   None: 0mm, Light: 0-2.5mm, Moderate: 2.5-7.6mm, Heavy: >7.6mm
df_features = df_features.withColumn(
    "precipitation_category",
    when(col("precipitation_mm").isNull(), "Unknown")
    .when(col("precipitation_mm") == 0, "None")
    .when(col("precipitation_mm") < 2.5, "Light")
    .when(col("precipitation_mm") < 7.6, "Moderate")
    .otherwise("Heavy")
)

# 5. is_rainy: Yağmurlu gün mü? (binary)
# İŞ MANTIĞI: Sınıflandırma problemi için basit hedef değişken kandidatı.
# "Yarın yağmur olacak mı?" tahmininde temel feature.
df_features = df_features.withColumn(
    "is_rainy",
    when(col("precipitation_mm") > 0, 1).otherwise(0)
)

# ============================================================
# FEATURE GROUP 3: ZAMAN-TABANLI FEATURE'LAR
# ============================================================

# 6. day_of_year: Yılın kaçıncı günü (1-366)
# İŞ MANTIĞI: Mevsimsel döngüleri yakalamak için. Sıcaklık ve yağış
# yıl içinde sinüzoidal bir patern izler — yılın günü bunu modele açar.
df_features = df_features.withColumn(
    "day_of_year",
    dayofyear(col("measurement_date"))
)

# 7. is_weekend: Hafta sonu mu?
# İŞ MANTIĞI: Doğa bilimi için zayıf görünebilir ama hava istasyonu
# bakım/kalibrasyon paterni iş günü vs hafta sonu farklılık gösterebilir.
df_features = df_features.withColumn(
    "day_of_week",
    dayofweek(col("measurement_date"))
).withColumn(
    "is_weekend",
    when(col("day_of_week").isin([1, 7]), 1).otherwise(0)
)

# 8. season_numeric: Sezonu sayısallaştır (model için)
# İŞ MANTIĞI: String "Winter/Spring..." yerine sayısal kodlama,
# tree-based modeller için daha verimli. Sıralı olduğu için ordinal encoding.
df_features = df_features.withColumn(
    "season_numeric",
    when(col("season") == "Winter", 1)
    .when(col("season") == "Spring", 2)
    .when(col("season") == "Summer", 3)
    .when(col("season") == "Autumn", 4)
    .otherwise(0)
)

print("✅ Yağış ve zaman feature'ları üretildi:")
print("   precipitation_category, is_rainy, day_of_year, is_weekend, season_numeric")

df_features.select("measurement_date", "season", "season_numeric", 
                   "day_of_year", "is_weekend",
                   "precipitation_mm", "precipitation_category", "is_rainy").show(5, truncate=False)

✅ Yağış ve zaman feature'ları üretildi:
   precipitation_category, is_rainy, day_of_year, is_weekend, season_numeric
+----------------+------+--------------+-----------+----------+----------------+----------------------+--------+
|measurement_date|season|season_numeric|day_of_year|is_weekend|precipitation_mm|precipitation_category|is_rainy|
+----------------+------+--------------+-----------+----------+----------------+----------------------+--------+
|1976-06-01      |Summer|3             |153        |0         |NULL            |Unknown               |0       |
|1976-06-04      |Summer|3             |156        |0         |NULL            |Unknown               |0       |
|1976-06-14      |Summer|3             |166        |0         |NULL            |Unknown               |0       |
|1976-06-16      |Summer|3             |168        |0         |NULL            |Unknown               |0       |
|1976-06-24      |Summer|3             |176        |0         |NULL            |Unknown     

In [4]:
from pyspark.sql.window import Window

# ============================================================
# FEATURE GROUP 4: ROLLING/LAG FEATURE'LAR (Gelişmiş)
# ============================================================
# Bu grup, geçmiş günlerin bilgisini bugünün satırına taşır.
# ML modeli "sıcaklık trendi" gibi temporal paternleri öğrenebilir.

# Şehir+tarih sıralı pencere tanımla
window_spec = Window.partitionBy("city_name").orderBy("measurement_date")

# 9. temp_lag_1: Bir gün önceki sıcaklık
# İŞ MANTIĞI: Bugünün sıcaklığı dünden çok kopuk değildir.
# Time-series tahmininde en güçlü feature genelde "bir önceki değer"dir.
df_features = df_features.withColumn(
    "temp_lag_1",
    lag("avg_temp_c", 1).over(window_spec)
)

# 10. temp_diff_from_yesterday: Düne göre sıcaklık farkı
# İŞ MANTIĞI: Hava değişim yönünü yakalar. Ani düşüş/yükseliş
# fırtına, soğuk dalga gibi olayların sinyali olabilir.
df_features = df_features.withColumn(
    "temp_diff_from_yesterday",
    col("avg_temp_c") - col("temp_lag_1")
)

# 11. temp_rolling_avg_7d: Son 7 günün ortalama sıcaklığı
# İŞ MANTIĞI: Günlük dalgalanmaları yumuşatır, gerçek trendi gösterir.
# "Bu hafta normalden sıcak mı?" sorusunu cevaplar.
window_7d = Window.partitionBy("city_name").orderBy("measurement_date").rowsBetween(-6, 0)
df_features = df_features.withColumn(
    "temp_rolling_avg_7d",
    avg("avg_temp_c").over(window_7d)
)

# 12. is_temp_anomaly: Bu sıcaklık 7 günlük ortalamadan çok mu uzak?
# İŞ MANTIĞI: Anomali tespiti için klasik feature.
# 7 günlük ortalamadan ±5°C farklıysa anormal kabul ediyoruz.
df_features = df_features.withColumn(
    "is_temp_anomaly",
    when(abs(col("avg_temp_c") - col("temp_rolling_avg_7d")) > 5, 1).otherwise(0)
)

print("✅ Lag ve rolling feature'lar üretildi:")
print("   temp_lag_1, temp_diff_from_yesterday, temp_rolling_avg_7d, is_temp_anomaly")

df_features.select("measurement_date", "avg_temp_c", "temp_lag_1",
                   "temp_diff_from_yesterday", "temp_rolling_avg_7d", 
                   "is_temp_anomaly").orderBy("measurement_date").show(10)

✅ Lag ve rolling feature'lar üretildi:
   temp_lag_1, temp_diff_from_yesterday, temp_rolling_avg_7d, is_temp_anomaly
+----------------+----------+----------+------------------------+-------------------+---------------+
|measurement_date|avg_temp_c|temp_lag_1|temp_diff_from_yesterday|temp_rolling_avg_7d|is_temp_anomaly|
+----------------+----------+----------+------------------------+-------------------+---------------+
|      1957-07-01|      27.0|      NULL|                    NULL|               27.0|              0|
|      1957-07-02|      22.8|      27.0|      -4.199999999999999|               24.9|              0|
|      1957-07-03|      24.3|      22.8|                     1.5|               24.7|              0|
|      1957-07-04|      26.6|      24.3|      2.3000000000000007| 25.174999999999997|              0|
|      1957-07-05|      30.8|      26.6|       4.199999999999999|               26.3|              0|
|      1957-07-06|      30.2|      30.8|     -0.6000000000000014|  

In [5]:
# ============================================================
# FEATURE TABLOSUNU DELTA LAKE'E KAYDET
# ============================================================

ML_FEATURES_PATH = "/home/jovyan/work/delta-lake/gold/ml_features"

# Feature dökümanı için sütun listesi
feature_columns = [
    # Temel kimlik
    "measurement_date", "city_name", "station_id", "season",
    # Ham değerler (ML modeli için referans)
    "avg_temp_c", "min_temp_c", "max_temp_c", "precipitation_mm",
    "year", "month", "day",
    # Üretilen feature'lar
    "temp_range_c",
    "is_freezing",
    "is_hot_day",
    "precipitation_category",
    "is_rainy",
    "day_of_year",
    "is_weekend",
    "season_numeric",
    "temp_lag_1",
    "temp_diff_from_yesterday",
    "temp_rolling_avg_7d",
    "is_temp_anomaly"
]

df_ml = df_features.select(*feature_columns)

# Delta Lake'e yaz
(df_ml.write
    .format("delta")
    .mode("overwrite")
    .save(ML_FEATURES_PATH))

print(f"✅ ML Features tablosu kaydedildi: {ML_FEATURES_PATH}")
print(f"   Toplam satır: {df_ml.count():,}")
print(f"   Toplam sütun: {len(df_ml.columns)}")
print(f"   Üretilen feature sayısı: 12")

print("\n=== ŞEMA ===")
df_ml.printSchema()

✅ ML Features tablosu kaydedildi: /home/jovyan/work/delta-lake/gold/ml_features
   Toplam satır: 5,000
   Toplam sütun: 23
   Üretilen feature sayısı: 12

=== ŞEMA ===
root
 |-- measurement_date: date (nullable = true)
 |-- city_name: string (nullable = true)
 |-- station_id: string (nullable = true)
 |-- season: string (nullable = true)
 |-- avg_temp_c: double (nullable = true)
 |-- min_temp_c: double (nullable = true)
 |-- max_temp_c: double (nullable = true)
 |-- precipitation_mm: double (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- day: integer (nullable = true)
 |-- temp_range_c: double (nullable = true)
 |-- is_freezing: integer (nullable = false)
 |-- is_hot_day: integer (nullable = false)
 |-- precipitation_category: string (nullable = false)
 |-- is_rainy: integer (nullable = false)
 |-- day_of_year: integer (nullable = true)
 |-- is_weekend: integer (nullable = false)
 |-- season_numeric: integer (nullable = false)
 |-- temp

In [6]:
# ============================================================
# FEATURE DOKÜMANTASYONU
# ============================================================
# Bu özet, Adım 6'da ML modelini kuracak kişi için referans niteliğinde.

import pandas as pd

feature_docs = [
    # (feature_adı, tipi, kategori, iş_mantığı)
    ("temp_range_c", "Numeric", "Sıcaklık", 
     "Günlük max-min sıcaklık farkı; iklim tipini gösterir"),
    ("is_freezing", "Binary", "Sıcaklık",
     "Donma noktası altı (avg<0); kar/don tahmini için"),
    ("is_hot_day", "Binary", "Sıcaklık",
     "Aşırı sıcak (avg>30); enerji/sağlık tahminleri için"),
    ("precipitation_category", "Categorical", "Yağış",
     "None/Light/Moderate/Heavy; meteorolojik standart eşikler"),
    ("is_rainy", "Binary", "Yağış",
     "Yağmur hedef değişkeni kandidatı"),
    ("day_of_year", "Numeric", "Zaman",
     "Yılın günü (1-366); mevsimsel döngüleri yakalar"),
    ("is_weekend", "Binary", "Zaman",
     "Hafta sonu (Cmt/Pzr); istasyon bakım paterni etkisi"),
    ("season_numeric", "Ordinal", "Zaman",
     "Sezon sayısallaştırma; tree-based modeller için"),
    ("temp_lag_1", "Numeric", "Temporal",
     "Bir gün önceki sıcaklık; en güçlü time-series feature"),
    ("temp_diff_from_yesterday", "Numeric", "Temporal",
     "Sıcaklık değişim yönü; fırtına/cephe sinyali"),
    ("temp_rolling_avg_7d", "Numeric", "Temporal",
     "7 günlük ortalama; trend yumuşatma"),
    ("is_temp_anomaly", "Binary", "Temporal",
     "7gün-ort'dan ±5°C uzaklık; anomali tespiti"),
]

docs_df = pd.DataFrame(feature_docs, columns=["Feature", "Tip", "Kategori", "İş Mantığı"])
print("=== ÜRETİLEN 12 FEATURE — DOKÜMANTASYON ===\n")
print(docs_df.to_string(index=False))

print("\n=== ÖZET İSTATİSTİKLER ===")
print(f"Toplam feature: 12")
print(f"  Sıcaklık tabanlı: 3 (temp_range_c, is_freezing, is_hot_day)")
print(f"  Yağış tabanlı: 2 (precipitation_category, is_rainy)")
print(f"  Zaman tabanlı: 3 (day_of_year, is_weekend, season_numeric)")
print(f"  Temporal/Lag: 4 (temp_lag_1, temp_diff_from_yesterday, temp_rolling_avg_7d, is_temp_anomaly)")
print(f"\nFeature tablosu: /delta-lake/gold/ml_features")
print(f"Adım 5 tamamlandı ✅")

=== ÜRETİLEN 12 FEATURE — DOKÜMANTASYON ===

                 Feature         Tip Kategori                                               İş Mantığı
            temp_range_c     Numeric Sıcaklık     Günlük max-min sıcaklık farkı; iklim tipini gösterir
             is_freezing      Binary Sıcaklık         Donma noktası altı (avg<0); kar/don tahmini için
              is_hot_day      Binary Sıcaklık      Aşırı sıcak (avg>30); enerji/sağlık tahminleri için
  precipitation_category Categorical    Yağış None/Light/Moderate/Heavy; meteorolojik standart eşikler
                is_rainy      Binary    Yağış                         Yağmur hedef değişkeni kandidatı
             day_of_year     Numeric    Zaman          Yılın günü (1-366); mevsimsel döngüleri yakalar
              is_weekend      Binary    Zaman      Hafta sonu (Cmt/Pzr); istasyon bakım paterni etkisi
          season_numeric     Ordinal    Zaman          Sezon sayısallaştırma; tree-based modeller için
              temp_lag_1    